In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install transformers datasets sentencepiece sacremoses accelerate evaluate sacrebleu -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 5.2 MB/s eta 0:00:00


In [ ]:
# Install required packages
!pip install transformers datasets sentencepiece sacremoses accelerate evaluate sacrebleu -q

import torch
from transformers import (
    MarianMTModel,
    MarianTokenizer,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)
from datasets import Dataset
import pandas as pd
import evaluate

# Check GPU availability
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# STEP 1: Load CSV Datasets
print("Loading datasets from CSV files...")

# Load the CSV files
train_df = pd.read_csv('/content/drive/MyDrive/SEMESTER 5/Tamil Inscription Reader/Translation - English to Tamil/Dataset/train_en_ta.csv')
val_df = pd.read_csv('/content/drive/MyDrive/SEMESTER 5/Tamil Inscription Reader/Translation - English to Tamil/Dataset/val_en_ta.csv')
test_df = pd.read_csv('/content/drive/MyDrive/SEMESTER 5/Tamil Inscription Reader/Translation - English to Tamil/Dataset/test_en_ta.csv')

print(f"Initial training samples: {len(train_df):,}")
print(f"Initial validation samples: {len(val_df):,}")
print(f"Initial test samples: {len(test_df):,}")

# CLEAN THE DATA
print("\nCleaning data...")

def clean_dataframe(df):
    df = df.dropna(subset=['english', 'tamil'])
    df['english'] = df['english'].astype(str).str.strip()
    df['tamil'] = df['tamil'].astype(str).str.strip()
    df = df[df['english'] != '']
    df = df[df['tamil'] != '']
    df = df[~df['english'].str.lower().isin(['nan', 'none'])]
    df = df[~df['tamil'].str.lower().isin(['nan', 'none'])]
    df = df[df['english'].str.len() >= 2]
    df = df[df['tamil'].str.len() >= 2]
    return df.reset_index(drop=True)

train_df = clean_dataframe(train_df)
val_df = clean_dataframe(val_df)
test_df = clean_dataframe(test_df)

print(f"\nAfter cleaning:")
print(f"  Training samples: {len(train_df):,}")
print(f"  Validation samples: {len(val_df):,}")
print(f"  Test samples: {len(test_df):,}")

print("\nSample data:")
print(train_df.head(3))

# Optional subset for faster training
train_df = train_df.head(100000)
val_df = val_df.head(10000)

print(f"\nUsing subset for training:")
print(f"  Training: {len(train_df):,} samples")
print(f"  Validation: {len(val_df):,} samples")

# Convert to Hugging Face Dataset
train_dataset = Dataset.from_pandas(train_df[['english', 'tamil']], preserve_index=False)
val_dataset = Dataset.from_pandas(val_df[['english', 'tamil']], preserve_index=False)
test_dataset = Dataset.from_pandas(test_df[['english', 'tamil']], preserve_index=False)

print("\nDatasets loaded successfully!")

# STEP 2: Load Pre-trained Model
model_name = "Helsinki-NLP/opus-mt-en-mul"

print(f"\nLoading model: {model_name}")

tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name).to(device)

print(f"Model loaded on {device}")

# STEP 3: Preprocessing
max_input_length = 128
max_target_length = 128

def preprocess_function(examples):
    inputs = examples["english"]
    targets = examples["tamil"]
    if isinstance(inputs, str):
        inputs = [inputs]
    if isinstance(targets, str):
        targets = [targets]
    inputs = [str(x).strip() if pd.notna(x) else "" for x in inputs]
    targets = [str(x).strip() if pd.notna(x) else "" for x in targets]
    valid_pairs = [(inp, tgt) for inp, tgt in zip(inputs, targets) if inp and tgt]
    if not valid_pairs:
        return {"input_ids": [], "attention_mask": [], "labels": []}
    inputs, targets = zip(*valid_pairs)
    inputs = list(inputs)
    targets = list(targets)
    model_inputs = tokenizer(
        inputs,
        max_length=max_input_length,
        truncation=True,
        padding="max_length"
    )
    labels = tokenizer(
        text_target=targets,
        max_length=max_target_length,
        truncation=True,
        padding="max_length"
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print("\nPreprocessing datasets...")

tokenized_train = train_dataset.map(preprocess_function, batched=True, remove_columns=['english', 'tamil'])
tokenized_val = val_dataset.map(preprocess_function, batched=True, remove_columns=['english', 'tamil'])

print("Preprocessing complete!")

# STEP 4: Training Configuration
print("\nConfiguring training parameters...")

training_args = Seq2SeqTrainingArguments(
    output_dir="./marian-en-ta-finetuned",
    eval_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=3,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    push_to_hub=False,
    logging_steps=500,
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="bleu",
    greater_is_better=True,
    report_to="none",
)

print(f"Training for {training_args.num_train_epochs} epochs")
print(f"Batch size: {training_args.per_device_train_batch_size}")
print(f"Learning rate: {training_args.learning_rate}")

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

metric = evaluate.load("sacrebleu")

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = [[l if l != -100 else tokenizer.pad_token_id for l in label] for label in labels]
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    decoded_labels = [[label] for label in decoded_labels]
    result = metric.compute(predictions=decoded_preds, references=decoded_labels)
    return {"bleu": result["score"]}

# STEP 5: Initialize Trainer
print("\nInitializing trainer...")

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Trainer initialized!")

# STEP 6: Train the Model
print("\nSTARTING TRAINING")
print("This may take a while depending on your dataset size...")

trainer.train()
print("Training complete!")

# STEP 7: Save the Model
print("\nSaving model...")

save_path = "/content/drive/MyDrive/SEMESTER 5/Tamil Inscription Reader/Translation - English to Tamil/Model/marian-en-ta-final_100k"
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print(f"Model saved to '{save_path}'")

# STEP 8: Evaluate on Test Set
print("\nEVALUATING ON TEST SET")

test_sample = test_df.head(1000)
test_sample_dataset = Dataset.from_pandas(test_sample[['english', 'tamil']], preserve_index=False)
tokenized_test = test_sample_dataset.map(preprocess_function, batched=True, remove_columns=['english', 'tamil'])

test_results = trainer.evaluate(tokenized_test)
print(f"Test BLEU Score: {test_results['eval_bleu']:.2f}")

# STEP 9: Test Translations
print("\nSAMPLE TRANSLATIONS")

def translate(text):
    model.eval()
    inputs = tokenizer(text, return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        translated = model.generate(**inputs, max_length=128, num_beams=5)
    return tokenizer.decode(translated[0], skip_special_tokens=True)

test_sentences = [
    "Hello, how are you?",
    "This project is Tamil Inscription Reader",
    "There are 5 big steps",
    "Alwas be happy",
    "Never loose hope",
]

for i, sentence in enumerate(test_sentences, 1):
    translation = translate(sentence)
    print(f"\n{i}. English: {sentence}")
    print(f"   Tamil:   {translation}")

print("\nTESTING WITH REAL DATA FROM TEST SET")

for i in range(min(5, len(test_df))):
    english = test_df.iloc[i]['english']
    actual_tamil = test_df.iloc[i]['tamil']
    predicted_tamil = translate(english)
    print(f"\n--- Example {i+1} ---")
    print(f"English:    {english}")
    print(f"Actual:     {actual_tamil}")
    print(f"Predicted:  {predicted_tamil}")

# STEP 10: Save Translation Function
print("\nTRANSLATION FUNCTION READY")
print("You can now use the translate() function to translate any English text!")
print("\nExample:")
print("  translation = translate('Good morning')")
print("  print(translation)")


print("\nALL DONE! Your model is ready to use!")


Using device: cuda
Loading datasets from CSV files...
Initial training samples: 173,538
Initial validation samples: 21,692
Initial test samples: 21,693

Cleaning data...


/tmp/ipython-input-2397500404.py:37: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['english'] = df['english'].astype(str).str.strip()
/tmp/ipython-input-2397500404.py:38: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['tamil'] = df['tamil'].astype(str).str.strip()



After cleaning:
  Training samples: 173,536
  Validation samples: 21,692
  Test samples: 21,693

Sample data:
                                             english  \
0  “So shall I seek the command other than that o...   
1  Use window class and window & role (specific w...   
2  Those upon whom the word will have come into e...   

                                               tamil  
0  (நபியே! கூறும்;) "அல்லாஹ் அல்லாதவனையா (தீர்ப்ப...  
1  சாளர தரம் மற்றும் சாளர & செயலை பயன்படுத்து (கு...  
2  எவர் மீது (அல்லாஹ்வின் தண்டனை பற்றிய) வாக்கு உ...  

Using subset for training:
  Training: 100,000 samples
  Validation: 10,000 samples

Datasets loaded successfully!

Loading model: Helsinki-NLP/opus-mt-en-mul


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

...

Epoch,Training Loss,Validation Loss,Bleu
1,0.458800,0.409036,12.733636
2,0.380900,0.357292,15.395528
3,0.349900,0.337140,16.901986


/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:4037: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 512, 'num_beams': 4, 'bad_words_ids': [[64109]]}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(
There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.encoder.embed_positions.weight', 'model.decoder.embed_tokens.weight', 'model.decoder.embed_positions.weight', 'lm_head.weight'].


Training complete!

Saving model...
Model saved to '/content/drive/MyDrive/SEMESTER 5/Tamil Inscription Reader/Translation - English to Tamil/Model/marian-en-ta-final_100k'

EVALUATING ON TEST SET


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Test BLEU Score: 17.39

SAMPLE TRANSLATIONS

1. English: Hello, how are you?
   Tamil:   வணக்கம், நீங்கள் எப்படி இருக்கிறீர்கள்?

2. English: This project is Tamil Inscription Reader
   Tamil:   இந்த திட்டம் தமிழ் இணைப்பு படிப்பாளர்

3. English: There are 5 big steps
   Tamil:   5 பெரிய நடவடிக்கைகள் உள்ளன

4. English: Alwas be happy
   Tamil:   அல்வாஸ் மகிழ்ச்சியாக இருக்கின்றார்

5. English: Never loose hope
   Tamil:   மிதமான நம்பிக்கை

TESTING WITH REAL DATA FROM TEST SET

--- Example 1 ---
English:    so that we bring down stones of clay upon them
Actual:     "அவர்கள் மீது (சுட்ட) களிமண் கற்களை எறிவதற்காக (நாங்கள் அனுப்பப்பட்டுள்ளோம்)-
Predicted:  ஆகவே, நாம் அவர்கள் மீது களிமண் கற்களை இறக்கி வைப்போம்.

--- Example 2 ---
English:    This is the status bar. It shows you the currently selected engine in the left corner.
Actual:     இது நிலைப்பெட்டி. தற்போது தேர்ந்தெடுத்த பொறியை இடது மூலையில் இது காண்பிக்கும்.
Predicted:  இது நிலைப்பட்டை. தற்போது தேர்ந்தெடுக்கப்பட்ட இயந்திரத்தை இடது கோ